# Martini: Bottom-Up Wasserstein Distance Optimization

This notebook optimizes Martini M2 force-field parameters via DiffTRe by
minimizing the Wasserstein (earth mover's) distance between simulated and
atomistic reference bond-distance and triplet-angle distributions.

## What is mythos?

**[Mythos](https://github.com/mythos-bio/mythos)** is a differentiable molecular simulation framework for parameterizing coarse-grained force fields. It provides automatic differentiation through simulation workflows — enabling gradient-based optimization of force field parameters to match experimental or all-atom reference data.

This notebook demonstrates bottom-up parameterization of a Martini coarse-grained membrane model. We match simulated bond-distance and triplet-angle distributions against all-atom reference distributions by minimizing the mean Wasserstein distance using GROMACS simulations coordinated via Ray.

## Imports

In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import MDAnalysis
import numpy as np
import optax
from mythos.energy.base import ComposedEnergyFunction
from mythos.energy.martini.base import MartiniTopology
from mythos.energy.martini.m2 import (
    LJ,
    Angle,
    AngleConfiguration,
    Bond,
    BondConfiguration,
    LJConfiguration,
)
from mythos.input.gromacs_input import read_params_from_topology
from mythos.observables.bond_distances import BondDistancesMapped
from mythos.observables.triplet_angles import TripletAnglesMapped
from mythos.observables.wasserstein import WassersteinDistanceMapped
from mythos.optimization.objective import DiffTReObjective
from mythos.optimization.optimization import RayOptimizer
from mythos.simulators.gromacs.gromacs import GromacsSimulator
from mythos.simulators.gromacs.utils import preprocess_topology
from mythos.ui.loggers.console import ConsoleLogger

jax.config.update("jax_enable_x64", True)

## Configuration

Adjust these settings for your system. The `INPUT_DIR` should contain `topol.top`, `md.mdp`, and `membrane.gro`.

In [ ]:
INPUT_DIR = Path("../../data/templates/martini/m2/DMPC/273K")
NUM_SIMS = 1
OPT_STEPS = 100
LEARNING_RATE = 5e-4
TEMPERATURE = 273.0
EQUILIBRATION_STEPS = 200_000
SIMULATION_STEPS = 500_000
SNAPSHOT_STEPS = 10_000
GROMACS_BINARY = None  # uses gmx on PATH

## Reference distributions
The bottom up observables fit to target distributions per bond and angle. Define
a map of bond/angle names to `.npy` files which contain the target distributions.
The format of these `.npy` files is a 1D array of occurrences of each bond
distance/angle value observed. Adjust these paths for your reference data.

In [ ]:

BOND_REFS = {
    "DMPC_C1A_C2A": "../../data/test-data/martini/DMPC_C1A-C2A_bond_dist.npy",
}
ANGLE_REFS = {
    "DMPC_C1A_C2A_C3A": "../../data/test-data/martini/DMPC_C1A-C2A-C3A_angle_dist.npy",
}
BOND_REF_UNITS = "angstrom"  # "nm" or "angstrom"
ANGLE_REF_UNITS = "radian"   # "radian" or "degree"

## Preprocess topology and build energy function

To build energy functions for martini we need to read parameters from a
preprocessed topology file (which expands any macros and includes all files).
Likewise we need to read the compiled topology which comes in the .tpr format
read into a mythose `MartiniTopology` object via MDAnalysis.

In [ ]:
preprocessed_top = INPUT_DIR / "preprocessed.top"
preprocessed_tpr = INPUT_DIR / "preprocessed.tpr"

if not preprocessed_top.exists() or not preprocessed_tpr.exists():
    preprocess_topology(INPUT_DIR, gromacs_binary=GROMACS_BINARY)

universe = MDAnalysis.Universe(preprocessed_tpr)
universe.transfer_to_memory()
parameters = read_params_from_topology(preprocessed_top)

top = MartiniTopology.from_universe(universe)

energy_fn = ComposedEnergyFunction(energy_fns=[
    LJ.from_topology(topology=top, params=LJConfiguration(**parameters["nonbond_params"])),
    Bond.from_topology(topology=top, params=BondConfiguration(**parameters["bond_params"])),
    Angle.from_topology(topology=top, params=AngleConfiguration(**parameters["angle_params"])),
])

## Load reference distributions and build Wasserstein observables

Adjusting units as necessary to the library convention of nm for distances and
radians for angles. The Wasserstein observables here use a mapped variant that
similary uses mapped variant (`WassersteinDistanceMapped`) of Bond/Angle
observables (`BondDistanceMapped`/`AngleDistanceMapped`) simplifying handling
multiple targets. Note that for large scale optimizations it may be desirable to
use the single-target variants (`WassersteinDistance` combined with
`BondDistance`, `AngleDistance`) to enable distribution of the computation
across multiple devices.

In [ ]:
bond_v_map = {}
for name, npy_file in BOND_REFS.items():
    ref = np.load(npy_file)
    if BOND_REF_UNITS == "angstrom":
        ref = ref * 0.1
    bond_v_map[name] = jnp.array(ref)

angle_v_map = {}
for name, npy_file in ANGLE_REFS.items():
    ref = np.load(npy_file)
    if ANGLE_REF_UNITS == "degree":
        ref = np.deg2rad(ref)
    angle_v_map[name] = jnp.array(ref)

wasserstein_observables = []
if bond_v_map:
    wasserstein_observables.append(WassersteinDistanceMapped(
        observable=BondDistancesMapped(topology=top, bond_names=tuple(bond_v_map.keys())),
        v_distribution_map=bond_v_map,
    ))
if angle_v_map:
    wasserstein_observables.append(WassersteinDistanceMapped(
        observable=TripletAnglesMapped(topology=top, angle_names=tuple(angle_v_map.keys())),
        v_distribution_map=angle_v_map,
    ))

n_total = sum(len(obs.v_distribution_map) for obs in wasserstein_observables)

## Create simulators and objective

Mythos supports the GROMACS engine via the `GromacsSimulator` class. We create
multiple simulators to run in parallel, which the `RayOptimizer` will
coordinate. The objective minimizes the RMSE of the Wasserstein distances across
all targets using DiffTre.

The GROMACS simulator handles equilibration by running the simulation without
output for a configurable number of steps. Overrides are supported via
`input_overrides`, which replaces values in the input `.mdp` file. Here we set
the output frequency (`nstxout`) and temperature (`ref_t`).

In [ ]:
simulators = GromacsSimulator.create_n(
    NUM_SIMS,
    input_dir=INPUT_DIR,
    energy_fn=energy_fn,
    equilibration_steps=EQUILIBRATION_STEPS,
    simulation_steps=SIMULATION_STEPS,
    input_overrides={"nstxout": SNAPSHOT_STEPS, "ref-t": TEMPERATURE},
    binary_path=GROMACS_BINARY,
)


def wasserstein_loss(traj, weights, *_):
    total = jnp.float64(0.0)
    for obs in wasserstein_observables:
        w_distances = obs(traj, weights)
        for v in w_distances.values():
            total = total + v
    loss = jnp.sqrt(total / n_total)
    return loss, (("wasserstein_mean", loss), ())


wasserstein_obj = DiffTReObjective(
    energy_fn=energy_fn,
    grad_or_loss_fn=wasserstein_loss,
    required_observables=[i for s in simulators for i in s.exposes()],
    name="WassersteinBottomUp",
)

## Run the optimization

In [ ]:
opt = RayOptimizer(
    simulators=simulators,
    objectives=[wasserstein_obj],
    optimizer=optax.adam(learning_rate=LEARNING_RATE),
    aggregate_grad_fn=lambda x: x[0],
    logger=ConsoleLogger(),
)

opt_params = energy_fn.opt_params()

print("\n=== Setup Complete ===")
print(f"Energy function terms: {[type(fn).__name__ for fn in energy_fn.energy_fns]}")
print(f"Number of parallel simulations: {NUM_SIMS}")
if bond_v_map:
    print(f"  Bond distributions: {list(bond_v_map.keys())}")
if angle_v_map:
    print(f"  Angle distributions: {list(angle_v_map.keys())}")
print(f"Optimizing {len(opt_params)} parameters")

output = opt.run(params=opt_params, n_steps=OPT_STEPS)
print("\nOptimization complete!")